# DetectIA — Fase 2: Entrenamiento CNN + Big Data (~357k) + ONNX (Google Colab GPU)

> **Objetivo:** Entrenar una Red Neuronal Convolucional (**CNN Residual**) sobre ~357.000 dígitos limpios (`MNIST`, `EMNIST Digits`, `QMNIST`, `USPS`) en `28 x 28` píxeles, con deduplicación SHA-256, *Data Augmentation* *on-the-fly*, monitoreo por épocas, gráficas de diagnóstico y exportación a **ONNX** para el laboratorio web.

---
### Pasos de ejecución en Google Colab:
1. Activa aceleración por GPU: **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4**.
2. (Opcional) Si quieres evaluar también contra tu `datos_test.npz` local y excluir sus muestras del entrenamiento, súbelo a la carpeta de archivos de Colab.
3. Ejecuta todo (**Ctrl + F9**). Al finalizar se descargará automáticamente `artefactos_detectia_cnn.zip`.

### Celda 1: Entorno, GPU y Dependencias ONNX

In [ ]:
!pip install -q onnx onnxruntime

import os
import json
import time
import hashlib
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import center_of_mass, shift
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import onnxruntime as ort

SEMILLA = 42
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[OK] PyTorch: {torch.__version__} | Dispositivo: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"     GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[AVISO] Sin GPU activa; el entrenamiento será más lento en CPU.")

### Celda 2: Descarga de Datasets (~357.000 muestras), Normalización MNIST y Deduplicación SHA-256
Unifica únicamente fuentes limpias compatibles en `28 x 28` (sin SVHN ni 8x8):
- **MNIST Train** (`60.000`)
- **EMNIST Digits Train** (`240.000`, corrigiendo orientación con `.transpose(0, 2, 1)`)
- **QMNIST `test50k`** (`50.000`)
- **USPS Train** (`7.291` escaladas de `16 x 16` a `28 x 28` con bounding box + Lanczos `20 x 20` + centro de masa en `(13.5, 13.5)`)

In [ ]:
def preprocesar_imagen_mnist(matriz_gris):
    img = Image.fromarray(matriz_gris.astype(np.uint8), mode='L')
    bbox = img.getbbox()
    if not bbox:
        return np.zeros((28, 28), dtype=np.uint8)
    recorte = img.crop(bbox)
    w, h = recorte.size
    if w > h:
        nuevo_w = 20
        nuevo_h = max(1, int(round((h * 20.0) / w)))
    else:
        nuevo_h = 20
        nuevo_w = max(1, int(round((w * 20.0) / h)))
    recorte_esc = recorte.resize((nuevo_w, nuevo_h), Image.Resampling.LANCZOS)
    canvas = Image.new('L', (28, 28), 0)
    canvas.paste(recorte_esc, ((28 - nuevo_w) // 2, (28 - nuevo_h) // 2))
    matriz = np.array(canvas, dtype=np.float32)
    max_val = np.max(matriz)
    if max_val > 0:
        matriz = (matriz / max_val) * 255.0
    if np.sum(matriz) > 0:
        cy, cx = center_of_mass(matriz)
        matriz = shift(matriz, [13.5 - cy, 13.5 - cx], order=1, mode='constant', cval=0.0)
    return np.clip(matriz, 0, 255).astype(np.uint8)

os.makedirs('data_raw', exist_ok=True)
os.makedirs('img', exist_ok=True)

print("Descargando MNIST, EMNIST Digits, QMNIST y USPS...")
ds_mnist_tr = datasets.MNIST('data_raw', train=True, download=True)
ds_mnist_te = datasets.MNIST('data_raw', train=False, download=True)

ds_emnist_tr = datasets.EMNIST('data_raw', split='digits', train=True, download=True)
ds_emnist_te = datasets.EMNIST('data_raw', split='digits', train=False, download=True)

ds_qmnist_50k = datasets.QMNIST('data_raw', what='test50k', download=True)

# Extraer matrices uint8 (N, 28, 28) y etiquetas int64
X_mnist_tr = ds_mnist_tr.data.numpy().astype(np.uint8)
y_mnist_tr = ds_mnist_tr.targets.numpy().astype(np.int64)
X_mnist_te = ds_mnist_te.data.numpy().astype(np.uint8)
y_mnist_te = ds_mnist_te.targets.numpy().astype(np.int64)

# EMNIST viene transpuesto por defecto -> corregir con .transpose(0, 2, 1)
X_emnist_tr = ds_emnist_tr.data.numpy().transpose(0, 2, 1).astype(np.uint8)
y_emnist_tr = ds_emnist_tr.targets.numpy().astype(np.int64)
X_emnist_te = ds_emnist_te.data.numpy().transpose(0, 2, 1).astype(np.uint8)
y_emnist_te = ds_emnist_te.targets.numpy().astype(np.int64)

X_qmnist_tr = ds_qmnist_50k.data.numpy().astype(np.uint8)
q_targets = ds_qmnist_50k.targets.numpy()
y_qmnist_tr = (q_targets[:, 0] if q_targets.ndim == 2 else q_targets).astype(np.int64)

# Construir hashes SHA-256 de todos los sets de evaluación para evitar data leakage
eval_sets = {
    'MNIST Test (10k)': (X_mnist_te, y_mnist_te),
    'EMNIST Test (40k)': (X_emnist_te, y_emnist_te),
}

fuentes = [
    ('MNIST Train', X_mnist_tr, y_mnist_tr),
    ('EMNIST Digits Train', X_emnist_tr, y_emnist_tr),
    ('QMNIST test50k', X_qmnist_tr, y_qmnist_tr),
]

# USPS es opcional si su servidor externo (csie.ntu.edu.tw) responde 502 Bad Gateway
try:
    ds_usps_tr = datasets.USPS('data_raw', train=True, download=True)
    ds_usps_te = datasets.USPS('data_raw', train=False, download=True)
    print("Normalizando USPS (16x16 -> 28x28 con centro de masa)...")
    X_usps_tr = np.array([preprocesar_imagen_mnist(img) for img in ds_usps_tr.data], dtype=np.uint8)
    y_usps_tr = np.array(ds_usps_tr.targets, dtype=np.int64)
    X_usps_te = np.array([preprocesar_imagen_mnist(img) for img in ds_usps_te.data], dtype=np.uint8)
    y_usps_te = np.array(ds_usps_te.targets, dtype=np.int64)
    fuentes.append(('USPS Train', X_usps_tr, y_usps_tr))
    eval_sets['USPS Test (2k)'] = (X_usps_te, y_usps_te)
except Exception as e:
    print(f"[AVISO] No se pudo descargar USPS ({e}); continuando con MNIST + EMNIST + QMNIST (~350k muestras).")

if os.path.exists('datos_test.npz'):
    npz_local = np.load('datos_test.npz')
    eval_sets['Local datos_test.npz'] = (npz_local['X_test'].reshape(-1, 28, 28).astype(np.uint8), npz_local['y_test'].astype(np.int64))

hashes_test = set()
for nombre_eval, (Xe, _) in eval_sets.items():
    for im in Xe:
        hashes_test.add(hashlib.sha256(im.tobytes()).hexdigest())

# Unificar y deduplicar entrenamiento

hashes_vistos = set()
X_pool, y_pool = [], []
conteo_fuentes = {}
duplicados_internos = 0
bloqueados_leakage = 0

for nombre_f, Xf, yf in fuentes:
    aceptados = 0
    for im, lbl in zip(Xf, yf):
        h = hashlib.sha256(im.tobytes()).hexdigest()
        if h in hashes_test:
            bloqueados_leakage += 1
            continue
        if h in hashes_vistos:
            duplicados_internos += 1
            continue
        hashes_vistos.add(h)
        X_pool.append(im)
        y_pool.append(lbl)
        aceptados += 1
    conteo_fuentes[nombre_f] = aceptados

X_pool = np.array(X_pool, dtype=np.uint8)
y_pool = np.array(y_pool, dtype=np.int64)

print(f"[OK] Pool limpio unificado: {len(X_pool):,} muestras.")
print(f"     Duplicados internos eliminados: {duplicados_internos:,} | Bloqueados por leakage vs Test: {bloqueados_leakage:,}")
for k, v in conteo_fuentes.items():
    print(f"     - {k}: {v:,}")

### Celda 3: Visualización del Dataset Unificado y Muestras de Data Augmentation

In [ ]:
# 1. Histograma de distribución de clases (0-9) y fuentes
clases_u, conteos_u = np.unique(y_pool, return_counts=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.bar(clases_u, conteos_u, color='#1a73e8', edgecolor='black')
ax1.set_title('Distribución de Clases en Pool Unificado (0 al 9)', fontweight='bold')
ax1.set_xlabel('Dígito')
ax1.set_ylabel('Cantidad de Muestras')
ax1.set_xticks(clases_u)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.barh(list(conteo_fuentes.keys()), list(conteo_fuentes.values()), color='#1e8e3e', edgecolor='black')
ax2.set_title('Aporte por Dataset Limpio', fontweight='bold')
ax2.set_xlabel('Muestras aceptadas')
ax2.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('img/cnn_distribucion_datos.png', dpi=150)
plt.show()

# 2. Muestra visual de orientación corregida + Data Augmentation (+-10 grados)
aug_preview = transforms.RandomAffine(degrees=10, translate=(0.08, 0.08), scale=(0.90, 1.10), shear=10)
fig, axes = plt.subplots(2, 8, figsize=(13, 3.8))
indices_muestra = np.linspace(0, len(X_pool) - 1, 8, dtype=int)
for col, idx in enumerate(indices_muestra):
    idx = min(max(0, idx), len(X_pool) - 1)
    img_orig = X_pool[idx]
    t_orig = torch.from_numpy(img_orig).unsqueeze(0).float() / 255.0
    t_aug = aug_preview(t_orig)
    axes[0, col].imshow(img_orig, cmap='gray')
    axes[0, col].set_title(f"Orig ({y_pool[idx]})", fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(t_aug.squeeze(0).numpy(), cmap='gray')
    axes[1, col].set_title("Augmentado", fontsize=9)
    axes[1, col].axis('off')

plt.suptitle('Orientación Corregida y Data Augmentation (+-10 deg, shear, escala)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('img/cnn_muestras_y_augmentation.png', dpi=150)
plt.show()
print("[OK] Gráficos iniciales guardados en 'img/'.")

### Celda 4: Partición Train / Validación y DataLoaders con Augmentation On-the-Fly

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_pool, y_pool, test_size=0.10, random_state=SEMILLA, stratify=y_pool
)

class MnistBigDataset(Dataset):
    def __init__(self, images_uint8, labels, augment=False):
        self.images = images_uint8
        self.labels = labels
        self.augment = augment
        self.affine = transforms.RandomAffine(
            degrees=10, translate=(0.08, 0.08), scale=(0.90, 1.10), shear=10
        )
        self.thicken = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).unsqueeze(0).float() / 255.0
        if self.augment:
            x = self.affine(x)
            if torch.rand(1).item() < 0.20:
                x = self.thicken(x.unsqueeze(0)).squeeze(0)
            x = torch.clamp(x, 0.0, 1.0)
        return x, int(self.labels[idx])

BATCH_SIZE = 256
train_loader = DataLoader(MnistBigDataset(X_tr, y_tr, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MnistBigDataset(X_val, y_val, augment=False), batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

print(f"[OK] Train: {len(X_tr):,} muestras | Validación: {len(X_val):,} muestras | Batch: {BATCH_SIZE}")

### Celda 5: Arquitectura de la CNN (Residual + BatchNorm + GAP)
Normaliza internamente `(x - 0.1307) / 0.3081` dentro de `forward()` para recibir directamente `[0.0, 1.0]` en `28 x 28`.

In [ ]:
class BloqueResidual(nn.Module):
    def __init__(self, in_c, out_c, stride=1, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.shortcut = nn.Sequential() if (stride == 1 and in_c == out_c) else nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_c)
        )

    def forward(self, x):
        out = F.silu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = F.silu(out + self.shortcut(x))
        return out

class DetectIACnnResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )
        self.layer1 = BloqueResidual(32, 32, stride=1, dropout=0.05)
        self.layer2 = BloqueResidual(32, 64, stride=2, dropout=0.10)
        self.layer3 = BloqueResidual(64, 128, stride=2, dropout=0.15)
        self.layer4 = BloqueResidual(128, 128, stride=1, dropout=0.15)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = (x - 0.1307) / 0.3081
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        return self.head(x)

modelo_cnn = DetectIACnnResNet().to(DEVICE)
n_params = sum(p.numel() for p in modelo_cnn.parameters())
print(f"[OK] Arquitectura DetectIACnnResNet lista ({n_params:,} parámetros).")

### Celda 6: Entrenamiento por Épocas (`AdamW` + `OneCycleLR` + `AMP` + `EarlyStopping`)

In [ ]:
MAX_EPOCAS = 25
PACIENCIA = 5
MIN_DELTA = 0.02  # Mejora mínima (%) en Val Acc para reiniciar la paciencia

criterio = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizador = torch.optim.AdamW(modelo_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizador, max_lr=3e-3, epochs=MAX_EPOCAS, steps_per_epoch=len(train_loader)
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

historial = []
mejor_val_acc = -1.0
mejor_val_para_paciencia = -1.0
mejor_epoca = 1
sin_mejora = 0

os.makedirs('modelos/cnn', exist_ok=True)
ruta_ckpt = 'modelos/cnn/best_cnn.pt'

t_inicio_total = time.time()
try:
    for epoca in range(1, MAX_EPOCAS + 1):
        t_ep = time.time()
        modelo_cnn.train()
        tr_loss_sum, tr_correct, tr_total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            optimizador.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                logits = modelo_cnn(xb)
                loss = criterio(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizador)
            scaler.update()
            scheduler.step()

            tr_loss_sum += loss.item() * len(yb)
            tr_correct += (logits.argmax(dim=1) == yb).sum().item()
            tr_total += len(yb)

        # Validación
        modelo_cnn.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    logits = modelo_cnn(xb)
                    loss = criterio(logits, yb)
                val_loss_sum += loss.item() * len(yb)
                val_correct += (logits.argmax(dim=1) == yb).sum().item()
                val_total += len(yb)

        tr_loss = tr_loss_sum / tr_total
        val_loss = val_loss_sum / val_total
        tr_acc = (tr_correct / tr_total) * 100.0
        val_acc = (val_correct / val_total) * 100.0
        lr_actual = optimizador.param_groups[0]['lr']

        historial.append({
            'epoca': epoca,
            'train_loss': round(tr_loss, 5),
            'val_loss': round(val_loss, 5),
            'train_acc': round(tr_acc, 4),
            'val_acc': round(val_acc, 4),
            'lr': round(lr_actual, 6)
        })

        if val_acc > mejor_val_acc:
            mejor_val_acc = val_acc
            mejor_epoca = epoca
            torch.save(modelo_cnn.state_dict(), ruta_ckpt)

        if val_acc > (mejor_val_para_paciencia + MIN_DELTA):
            mejor_val_para_paciencia = val_acc
            sin_mejora = 0
        else:
            sin_mejora += 1

        print(f"Época {epoca:2d}/{MAX_EPOCAS} ({time.time()-t_ep:.1f}s) | Train Acc: {tr_acc:.2f}% | Val Acc: {val_acc:.2f}% | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {lr_actual:.5f}")

        if sin_mejora >= PACIENCIA:
            print(f"[EARLY STOPPING] Detenido automáticamente en época {epoca} tras {PACIENCIA} épocas sin mejora >= {MIN_DELTA}%.")
            break
except KeyboardInterrupt:
    print(f"\n[DETENIDO MANUALMENTE] Conservando el mejor modelo guardado (Época #{mejor_epoca}: {mejor_val_acc:.2f}%).")

tiempo_total_train = time.time() - t_inicio_total
if os.path.exists(ruta_ckpt):
    modelo_cnn.load_state_dict(torch.load(ruta_ckpt, map_location=DEVICE))
print(f"\n[OK] Entrenamiento finalizado en {tiempo_total_train:.1f}s. Mejor Val Acc: {mejor_val_acc:.2f}% (Época #{mejor_epoca})")

### Celda 7: Curvas de Diagnóstico por Época (Loss, Accuracy, Brecha Overfitting y Learning Rate)

In [ ]:
ep_x = [h['epoca'] for h in historial]
tr_l = [h['train_loss'] for h in historial]
va_l = [h['val_loss'] for h in historial]
tr_a = [h['train_acc'] for h in historial]
va_a = [h['val_acc'] for h in historial]
gaps = [t - v for t, v in zip(tr_a, va_a)]
lrs = [h['lr'] for h in historial]

fig, axs = plt.subplots(2, 2, figsize=(13, 9))

axs[0, 0].plot(ep_x, tr_l, 'o-', color='#1a73e8', label='Train Loss')
axs[0, 0].plot(ep_x, va_l, 's-', color='#d93025', label='Val Loss')
axs[0, 0].axvline(mejor_epoca, color='#e37400', linestyle='--', label=f'Mejor (Época {mejor_epoca})')
axs[0, 0].set_title('Pérdida por Época (Loss)', fontweight='bold')
axs[0, 0].set_xlabel('Época'); axs[0, 0].set_ylabel('CrossEntropy Loss')
axs[0, 0].grid(True, linestyle='--', alpha=0.5); axs[0, 0].legend()

axs[0, 1].plot(ep_x, tr_a, 'o-', color='#1a73e8', label='Train Accuracy')
axs[0, 1].plot(ep_x, va_a, 's-', color='#1e8e3e', label='Val Accuracy')
axs[0, 1].axvline(mejor_epoca, color='#e37400', linestyle='--', label=f'Mejor ({mejor_val_acc:.2f}%)')
axs[0, 1].set_title('Exactitud por Época (Train vs. Val)', fontweight='bold')
axs[0, 1].set_xlabel('Época'); axs[0, 1].set_ylabel('Accuracy (%)')
axs[0, 1].grid(True, linestyle='--', alpha=0.5); axs[0, 1].legend()

axs[1, 0].plot(ep_x, gaps, 'o-', color='#8e24aa', label='Gap (Train - Val)')
axs[1, 0].axhline(0, color='gray', linestyle=':', linewidth=1)
axs[1, 0].set_title('Brecha de Generalización / Overfitting', fontweight='bold')
axs[1, 0].set_xlabel('Época'); axs[1, 0].set_ylabel('Puntos porcentuales')
axs[1, 0].grid(True, linestyle='--', alpha=0.5); axs[1, 0].legend()

axs[1, 1].plot(ep_x, lrs, 'o-', color='#00897b', label='Learning Rate')
axs[1, 1].set_title('Planificación OneCycleLR', fontweight='bold')
axs[1, 1].set_xlabel('Época'); axs[1, 1].set_ylabel('LR')
axs[1, 1].grid(True, linestyle='--', alpha=0.5); axs[1, 1].legend()

plt.suptitle('DetectIA Fase 2 — Diagnóstico por Épocas (CNN Residual)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('img/cnn_curvas_epocas.png', dpi=150)
plt.show()
print("[OK] Guardada en 'img/cnn_curvas_epocas.png'.")

### Celda 8: Evaluación Multi-Dataset y Matriz de Confusión

In [ ]:
def evaluar_numpy(modelo, X_u8, batch=512):
    modelo.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_u8), batch):
            xb = torch.from_numpy(X_u8[i:i+batch]).unsqueeze(1).float().to(DEVICE) / 255.0
            preds.append(modelo(xb).argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)

benchmarks = {}
pred_mnist = None
for nombre_eval, (Xe, ye) in eval_sets.items():
    t0 = time.time()
    yp = evaluar_numpy(modelo_cnn, Xe)
    dt = time.time() - t0
    acc = accuracy_score(ye, yp) * 100.0
    benchmarks[nombre_eval] = {
        'muestras': int(len(ye)),
        'aciertos': int(np.sum(yp == ye)),
        'errores': int(np.sum(yp != ye)),
        'accuracy_pct': round(float(acc), 2),
        'tiempo_seg': round(dt, 2)
    }
    print(f" - {nombre_eval:<22}: {acc:.2f}% ({np.sum(yp == ye)}/{len(ye)})")
    if nombre_eval.startswith('MNIST Test'):
        pred_mnist = yp

# Matriz de Confusión sobre MNIST Test
conf = confusion_matrix(y_mnist_te, pred_mnist)
plt.figure(figsize=(8.5, 6.5))
plt.imshow(conf, cmap=plt.cm.Blues)
plt.title(f"Matriz de Confusión CNN — MNIST Test ({benchmarks['MNIST Test (10k)']['accuracy_pct']:.2f}%)", fontweight='bold')
plt.colorbar()
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
thresh = conf.max() / 2.0
for r in range(10):
    for c in range(10):
        v = conf[r, c]
        plt.text(c, r, f"{v}", ha='center', va='center', color='white' if v > thresh else 'black', fontsize=9)
plt.xlabel('Predicción'); plt.ylabel('Real')
plt.tight_layout()
plt.savefig('img/cnn_matriz_confusion.png', dpi=150)
plt.show()

# Gráfico comparativo por dataset
plt.figure(figsize=(8, 4.2))
nombres_b = list(benchmarks.keys())
accs_b = [benchmarks[k]['accuracy_pct'] for k in nombres_b]
bars = plt.bar(nombres_b, accs_b, color=['#1a73e8', '#1e8e3e', '#f9ab00', '#8e24aa'][:len(nombres_b)])
for b, val in zip(bars, accs_b):
    plt.text(b.get_x() + b.get_width()/2, val + 0.3, f"{val:.2f}%", ha='center', fontweight='bold')
plt.ylim(min(accs_b) - 3, 101)
plt.title('Exactitud de la CNN en Distintos Conjuntos de Prueba', fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('img/cnn_benchmarks_datasets.png', dpi=150)
plt.show()

### Celda 9: Exportación a ONNX, Metadatos y Descarga de `artefactos_detectia_cnn.zip`

In [ ]:
modelo_cpu = modelo_cnn.cpu().eval()
ruta_onnx = 'modelos/cnn/modelo_cnn.onnx'
dummy_input = torch.randn(1, 1, 28, 28, dtype=torch.float32)

export_kwargs = dict(
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17
)
try:
    torch.onnx.export(modelo_cpu, dummy_input, ruta_onnx, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(modelo_cpu, dummy_input, ruta_onnx, **export_kwargs)

# Verificación PyTorch vs ONNX Runtime
sesion = ort.InferenceSession(ruta_onnx, providers=['CPUExecutionProvider'])
muestra_test = (X_mnist_te[:16].astype(np.float32) / 255.0)[:, None, :, :]
with torch.no_grad():
    out_pt = modelo_cpu(torch.from_numpy(muestra_test)).numpy()
out_ort = sesion.run(None, {'input': muestra_test})[0]
max_diff = float(np.max(np.abs(out_pt - out_ort)))
print(f"[OK] Exportado '{ruta_onnx}' | Diferencia máxima PyTorch vs ONNX: {max_diff:.6f}")
assert max_diff < 1e-3, f"Diferencia excesiva en exportación ONNX: {max_diff}"

# Guardar historial_epocas.json, metricas_cnn.json e info.json
ruta_hist = 'modelos/cnn/historial_epocas.json'
with open(ruta_hist, 'w', encoding='utf-8') as f:
    json.dump({
        'modelo': 'DetectIA CNN Residual',
        'mejor_epoca': int(mejor_epoca),
        'mejor_val_acc': round(float(mejor_val_acc), 2),
        'epocas': historial
    }, f, indent=2, ensure_ascii=False)

acc_principal = benchmarks.get('Local datos_test.npz', benchmarks['MNIST Test (10k)'])['accuracy_pct']
ruta_metricas = 'modelos/cnn/metricas_cnn.json'
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    json.dump({
        'modelo': 'CNN Residual (PyTorch -> ONNX)',
        'total_pool_entrenamiento': int(len(X_pool)),
        'tiempo_entrenamiento_seg': round(tiempo_total_train, 2),
        'mejor_epoca': int(mejor_epoca),
        'mejor_val_acc': round(float(mejor_val_acc), 2),
        'benchmarks': benchmarks
    }, f, indent=2, ensure_ascii=False)

ruta_info = 'modelos/cnn/info.json'
with open(ruta_info, 'w', encoding='utf-8') as f:
    json.dump({
        'id': 'cnn',
        'nombre': 'Red Neuronal CNN',
        'familia': 'CNN Residual (ONNX)',
        'tipo': 'onnx',
        'archivo_modelo': 'modelo_cnn.onnx',
        'archivo_metricas': 'metricas_cnn.json',
        'archivo_historial': 'historial_epocas.json',
        'accuracy_pct': acc_principal,
        'normalizado': True
    }, f, indent=2, ensure_ascii=False)

# Empaquetar en artefactos_detectia_cnn.zip
zip_salida = 'artefactos_detectia_cnn.zip'
with zipfile.ZipFile(zip_salida, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [ruta_onnx, ruta_info, ruta_metricas, ruta_hist]:
        z.write(p, arcname=p)
    for img_name in sorted(os.listdir('img')):
        ruta_img = os.path.join('img', img_name)
        if os.path.isfile(ruta_img):
            z.write(ruta_img, arcname=os.path.join('modelos/cnn/img', img_name))

print(f"[OK] Paquete creado: {zip_salida} ({os.path.getsize(zip_salida)/(1024*1024):.2f} MB)")
from google.colab import files
files.download(zip_salida)